In [3]:
import vertica_python
from getpass import getpass


def try_select(s):
    conn_info = {'host': '51.250.75.20', 
             'port': '5433',
             'user': 'goel52_yandex_ru',       
             'password': '5SaZNh4qa7',
             'database': 'dwh',
             # Вначале он нам понадобится, а дальше — решите позже сами
             'autocommit': True
}

    with vertica_python.connect(**conn_info) as conn:
        # Select 1 — ваш код здесь; 
        сur = conn.cursor()
        сur.execute(s)
        res = сur.fetchall() 
        return res 

In [4]:
try_select("Select 1 as a1;")

/Users/oleg/yandex_pracktikum/venv/lib/python3.13/site-packages/vertica_python/vertica/connection.py:629: UserWarning: TLS is not configured on the server. Proceeding with an unencrypted channel.
HINT: Set connection option 'tlsmode' to 'disable' to explicitly create a non-TLS connection.
  warnings.warn(msg + hint)


[[1]]

In [4]:
try_select("""CREATE TABLE BAD_IDEA (
    i int primary key,
    v varchar(16)
);  """)

DuplicateObject: Severity: ROLLBACK, Message: Object "BAD_IDEA" already exists, Sqlstate: 42710, Routine: outputAlreadyExistsEreport, File: /data/jenkins/workspace/RE-ReleaseBuilds/RE-Jackhammer/server/vertica/Commands/DDL.cpp, Line: 24919, Error Code: 4213, SQL: 'CREATE TABLE BAD_IDEA (     i int primary key,     v varchar(16) );  '

In [5]:
try_select("""ALTER TABLE BAD_IDEA SET MERGOUT 0;""")

VerticaSyntaxError: Severity: ERROR, Message: Syntax error at or near "MERGOUT", Sqlstate: 42601, Position: 26, Routine: base_yyerror, File: /data/jenkins/workspace/RE-ReleaseBuilds/RE-Jackhammer/server/vertica/Parser/scan.l, Line: 1051, Error Code: 4856, SQL: 'ALTER TABLE BAD_IDEA SET MERGOUT 0;'

In [7]:
%%time
N = 10000
batch = 5

conn_info = {'host': '51.250.75.20', 
             'port': '5433',
             'user': 'goel52_yandex_ru',       
             'password': '5SaZNh4qa7',
             'database': 'dwh',
             # Вначале он нам понадобится, а дальше — решите позже сами
             'autocommit': True
}

with vertica_python.connect(**conn_info) as conn:
    curs = conn.cursor()
    insert_stmt = 'INSERT INTO BAD_IDEA VALUES ({},\'a\');'
    
    for i in range(0, N, batch):
        # будем отправлять сразу по несколько команд
        curs.execute(
            '\n'.join(
                [insert_stmt.format(i + j) for j in range(batch)])
        )
        
    conn.commit()

InsufficientResources: Severity: ERROR, Message: Too many ROS containers exist for the following projections: 
GOEL52_YANDEX_RU.BAD_IDEA_b0 (limit = 3938, ROS files = 3936, DV files = 0, new files = 3), Sqlstate: 53000, Hint: Please wait for the tuple mover to catch up.  Use 'select * from v_monitor.tuple_mover_operations;' to monitor.
Also, please check if mergeout is disabled for this table. Use 'select * from v_catalog.tables where is_mergeout_enabled='f';', Routine: Exec_preparePlanForLaunch, File: /data/jenkins/workspace/RE-ReleaseBuilds/RE-Jackhammer/server/vertica/Dist/PlanExecCoordinator.cpp, Line: 2723, Error Code: 5065, SQL: "INSERT INTO BAD_IDEA VALUES (3860,'a'); INSERT INTO BAD_IDEA VALUES (3861,'a'); INSERT INTO BAD_IDEA VALUES (3862,'a'); INSERT INTO BAD_IDEA VALUES (3863,'a'); INSERT INTO BAD_IDEA VALUES (3864,'a');"

In [31]:
import csv
from pathlib import Path

dataset = 'test_dataset.csv'
N = 10000 # на этот раз можете поставить даже 10 млн

with open(dataset, 'w') as csvfile:
    fwriter = csv.writer(csvfile, delimiter='|')
    for i in range(N):
        fwriter.writerow([i, 'asds'])

# эта команда напечатает абсолютный путь к файлу, скопируйте его
print(Path(dataset).resolve())

# а это пара первых строк для визуализации результата:
with open(dataset, 'r') as csvfile:
    for i in range(5):
        print(csvfile.readline(), end='')

/Users/16997512/yandex_pracktikum/sprint_6/test_dataset.csv
0|asds
1|asds
2|asds
3|asds
4|asds


In [4]:
import boto3

AWS_ACCESS_KEY_ID = "<YC_ACCESS_KEY_ID>"
AWS_SECRET_ACCESS_KEY = "<YC_SECRET_ACCESS_KEY>"

session = boto3.session.Session()
s3_client = session.client(
    service_name='s3',
    endpoint_url='https://storage.yandexcloud.net',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
s3_client.download_file(
    Bucket='sprint6',
    Key='groups.csv',
    Filename='groups.csv'
) 

In [5]:
!ls

PracticaumSp5MongoDb.crt group_log.csv            sprint-6-tests
Untitled.ipynb           groups.csv
__init__.py              s6


In [15]:
!pwdimport sys
!{sys.executable} -m pip install pendulum


[notice] A new release of pip is available: 23.3.1 -> 23.3.2
[notice] To update, run: /usr/local/bin/python3.11 -m pip install --upgrade pip


In [13]:
def fetch_s3_file(bucket: str, key: str) -> str:
    session = boto3.session.Session()
    s3_client = session.client(
        service_name='s3',
        endpoint_url='https://storage.yandexcloud.net',
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
    s3_client.download_file(
				Bucket=bucket, 
				Key=key, 
				Filename=f'/data/{key}'
)

In [14]:
fetch_s3_file('sprint6', 'groups.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/data/groups.csv.fC15BAD2'

In [12]:
session = boto3.session.Session()
s3_client = session.client(
    service_name='s3',
    endpoint_url='https://storage.yandexcloud.net',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
s3_client.download_file(
            Bucket='sprint6',
            Key=key,
            Filename='/data/groups.csv'

SyntaxError: incomplete input (642895007.py, line 11)

In [27]:
!ls -al

total 96
drwxrwxrwx@  9 oleg  staff    288 Jan 24 20:51 .
drwxrwxrwx@ 32 oleg  staff   1024 Jan 24 20:35 ..
-rwxrwxrwx@  1 oleg  staff   6148 Nov  8 01:03 .DS_Store
drwxr-xr-x   3 oleg  staff     96 Jan 24 20:24 .ipynb_checkpoints
-rwxrwxrwx@  1 oleg  staff   3579 Jun 20  2023 PracticaumSp5MongoDb.crt
-rwxrwxrwx@  1 oleg  staff  36817 Jan 24 20:50 Untitled.ipynb
-rwxrwxrwx@  1 oleg  staff      0 Aug 18 20:28 __init__.py
drwxrwxrwx@ 16 oleg  staff    512 Aug 16 22:28 s6-lessons
drwxrwxrwx@ 49 oleg  staff   1568 Aug 18 20:29 sprint-6-tests


In [28]:
import boto3
import pendulum


AWS_ACCESS_KEY_ID = "<YC_ACCESS_KEY_ID>"
AWS_SECRET_ACCESS_KEY = "<YC_SECRET_ACCESS_KEY>"

def fetch_s3_file(bucket: str, key: str) -> str:
    session = boto3.session.Session()
    s3_client = session.client(
        service_name='s3',
        endpoint_url='https://storage.yandexcloud.net',
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
    s3_client.download_file(
				Bucket=bucket, 
				Key=key, 
				Filename=f'{key}'
)

fetch_s3_file('sprint6', 'group_log.csv')




In [29]:
!ls -al

total 63584
drwxrwxrwx@ 10 oleg  staff       320 Jan 24 20:51 .
drwxrwxrwx@ 32 oleg  staff      1024 Jan 24 20:35 ..
-rwxrwxrwx@  1 oleg  staff      6148 Nov  8 01:03 .DS_Store
drwxr-xr-x   3 oleg  staff        96 Jan 24 20:24 .ipynb_checkpoints
-rwxrwxrwx@  1 oleg  staff      3579 Jun 20  2023 PracticaumSp5MongoDb.crt
-rwxrwxrwx@  1 oleg  staff     36817 Jan 24 20:50 Untitled.ipynb
-rwxrwxrwx@  1 oleg  staff         0 Aug 18 20:28 __init__.py
-rw-r--r--   1 oleg  staff  32228323 Jan 24 20:51 group_log.csv
drwxrwxrwx@ 16 oleg  staff       512 Aug 16 22:28 s6-lessons
drwxrwxrwx@ 49 oleg  staff      1568 Aug 18 20:29 sprint-6-tests


In [8]:
import boto3
# import pendulum


AWS_ACCESS_KEY_ID = "<YC_ACCESS_KEY_ID>"
AWS_SECRET_ACCESS_KEY = "<YC_SECRET_ACCESS_KEY>"

def fetch_s3_file(bucket: str, key: str) -> str:
    session = boto3.session.Session()
    s3_client = session.client(
        service_name='s3',
        endpoint_url='https://storage.yandexcloud.net',
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
    s3_client.download_file(
				Bucket=bucket, 
				Key=key, 
				Filename=f'{key}'
)

fetch_s3_file('sprint6', 'group_log.csv')

In [9]:
!ls

PracticaumSp5MongoDb.crt __init__.py              s6
Untitled.ipynb           group_log.csv            sprint-6-tests


In [5]:
!rm group_log.csv

In [1]:
202049/245

824.6897959183674

In [2]:
183604/216

850.0185185185185

In [3]:
216977/194

1118.4381443298969

In [4]:
157629/226

697.4734513274336

In [5]:
176806/276

640.6014492753624